# FREUID Challenge — Diagnostic: `is_digital` Test-Time Flag - v5

**Purpose:** `test_df` in v1/v2/v4 hardcodes `is_digital=0` for every test row (public *and*
private). Training data is 99.97% `is_digital=1` (69,332 of 69,352 rows) — the OOF breakdown from
the v4 run showed `is_digital=0` (recaptured) FREUID = 0.0872 vs `is_digital=1` (digital) FREUID =
0.00046, a ~190x gap, on only 20 training examples of the recaptured class. It's plausible a
meaningful share of the CV/LB gap (OOF FREUID 0.0005 vs public LB 0.22090) comes from feeding
every test image through the model in a regime it has barely seen, rather than from a deeper
generalization failure of the vision backbone.

**This notebook is INFERENCE-ONLY.** It loads the already-trained fold0/fold1 checkpoints from the
v4 run and does not touch model weights or training code — safe to run even after the July 13
code-freeze (documented inference-time flag, same spirit as the `VARIANT` env var pattern
discussed on the competition forum).

**LB-isolation caveat (from Discussions.docx):** the host confirmed the public leaderboard score
is affected by values in the private-row (`fill_value`) portion of the submission file too, not
just the public rows. So this notebook uses the **exact same `fill_value`** as the last real
submission (`fraud_rate` computed from `train_labels.csv`, ≈0.4232) for every submission it
produces here. If the LB score moves after submitting, that movement is attributable to the real
7,821 public-row predictions changing — not to a shifted fill value.

**What this notebook produces:**
1. `submission_diag_isdigital1_ensemble.csv` — real predictions with `is_digital=1` for all test rows
2. `submission_diag_isdigital0_baseline.csv` — replica of the original v4 submission (`is_digital=0`), for local diffing only
3. A local diff report (score deltas, correlation, threshold-crossing count) so you have an offline signal before spending a Kaggle submission

**Only submit file (1) to Kaggle.** File (2) reproduces what you already submitted — submitting it
again wastes a daily slot for no new information.


## 1. Environment Setup

In [ ]:
import os
import time
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['HF_HUB_OFFLINE']           = '1'
os.environ['TRANSFORMERS_OFFLINE']     = '1'

import numpy as np
import pandas as pd
import scipy.optimize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.isotonic import IsotonicRegression
from PIL import Image

warnings.filterwarnings('ignore')

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'timm   : {timm.__version__}')

## 2. Config

Must match the v4 training config exactly for architecture fields (`BACKBONE`, `IMG_SIZE`, `DROP_RATE`, `USE_METADATA`, `DOC_EMB_DIM`) so the checkpoint `state_dict` loads cleanly. `N_DOC_TYPES` is recomputed from `train_labels.csv` in Section 4 using the identical vocab-building logic as v4, so it will match automatically as long as the labels file hasn't changed.

In [ ]:
@dataclass
class CFG:
    DATA_DIR:          str   = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
    WEIGHTS_PATH:      str   = ('/kaggle/input/models/timm/tf-efficientnet/'
                                'pytorch/tf-efficientnet-b4/1/'
                                'tf_efficientnet_b4_aa-818f208c.pth')
    OUTPUT_DIR:        str   = '/kaggle/working'
    CHECKPOINT_DIR:    str   = '/kaggle/working/checkpoints'
    USE_FULL_DATA:     bool  = True
    TRAIN_LABELS_FILE: str   = ''
    TEST_IMG_SUBDIR:   str   = 'public_test'

    # Must match v4 training architecture exactly
    BACKBONE:          str   = 'tf_efficientnet_b4'
    PRETRAINED:        bool  = True
    IMG_SIZE:          int   = 320
    DROP_RATE:         float = 0.3
    USE_METADATA:      bool  = True
    N_DOC_TYPES:       int   = 256   # placeholder — recomputed in Section 4
    DOC_EMB_DIM:       int   = 16

    # Inference-only knobs
    BATCH_SIZE:        int   = 16
    NUM_WORKERS:       int   = 0
    PIN_MEMORY:        bool  = True
    AMP:               bool  = True
    CALIB_METHOD:      str   = 'temperature'   # must match how each fold was calibrated

    def __post_init__(self):
        if not self.TRAIN_LABELS_FILE:
            self.TRAIN_LABELS_FILE = (
                'train_labels.csv' if self.USE_FULL_DATA
                else 'train_sample_labels.csv'
            )


CFG = CFG()
Path(CFG.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded (inference-only).')
print(f'  IMG_SIZE   : {CFG.IMG_SIZE}')
print(f'  BACKBONE   : {CFG.BACKBONE}')

## 3. Checkpoint Restore

Same pattern as v4 Section 3. If this notebook is run in the **same Kaggle session** right after v4 training, checkpoints are already in `CHECKPOINT_DIR` and this cell no-ops. Otherwise, edit `CKPT_SOURCE_CANDIDATES` to point at the attached dataset containing your v4 output (`fold0_best.pth`, `fold1_best.pth`).

In [ ]:
import shutil

CKPT_SOURCE_CANDIDATES = [
    '/kaggle/input/datasets/maheshwarmishra/v4fold0',
    '/kaggle/input/datasets/maheshwarmishra/v4fold1',
      # e.g. v4's own committed output, if attached as a dataset
]

existing = list(Path(CFG.CHECKPOINT_DIR).glob('fold*_best.pth'))
if existing:
    print(f'CHECKPOINT_DIR already populated (same-session continuation): '
          f'{[p.name for p in existing]}')
else:
    copied = []
    for src_dir in CKPT_SOURCE_CANDIDATES:
        src_path = Path(src_dir)
        if not src_path.exists():
            continue
        for ckpt_file in src_path.glob('fold*_best.pth'):
            dest = Path(CFG.CHECKPOINT_DIR) / ckpt_file.name
            shutil.copy2(ckpt_file, dest)
            copied.append(ckpt_file.name)
    print(f'Checkpoints copied into {CFG.CHECKPOINT_DIR}: {copied}')
    if not copied:
        print('WARNING: no checkpoint files found. Check the Input sidebar for the exact '
              'mounted dataset path and update CKPT_SOURCE_CANDIDATES above.')

print('Current contents:', sorted(os.listdir(CFG.CHECKPOINT_DIR)))

## 4. Data Loading

Loads `train_labels.csv` only to rebuild the document-type vocabulary (needed so `CFG.N_DOC_TYPES` matches what the checkpoint's embedding layer expects — `type_idx` itself is always `<UNK>`/0 for test rows regardless, unchanged from v4). Then builds two `test_df` variants that differ **only** in `is_digital`.

In [ ]:
IMAGE_EXTENSIONS = ('.jpeg', '.jpg', '.png', '.webp', '.bmp')

def find_images_in_dir(directory: Path) -> List[Path]:
    files = []
    for ext in IMAGE_EXTENSIONS:
        files += list(directory.glob(f'*{ext}'))
    return sorted(files)


def find_images_robust(base_dir: Path, subdir: str) -> Tuple[Path, List[Path]]:
    """Identical logic to v2/v4 — handles the doubled-folder Kaggle zip quirk."""
    flat = base_dir / subdir
    files = find_images_in_dir(flat) if flat.exists() else []
    if files:
        return flat, files
    doubled = base_dir / subdir / subdir
    files = find_images_in_dir(doubled) if doubled.exists() else []
    if files:
        return doubled, files
    if flat.exists():
        for child in sorted(flat.iterdir()):
            if child.is_dir():
                files = find_images_in_dir(child)
                if files:
                    return child, files
    return flat, []


def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    c = base_dir / rel_path
    if c.exists(): return c
    parts = Path(rel_path).parts
    if len(parts) > 1:
        d = base_dir / parts[0] / rel_path
        if d.exists(): return d
    f = base_dir / Path(rel_path).name
    if f.exists(): return f
    return c


data_dir = Path(CFG.DATA_DIR)

# Train labels — needed ONLY to rebuild vocab size and fraud_rate (fill_value).
labels_path = data_dir / CFG.TRAIN_LABELS_FILE
if not labels_path.exists():
    raise FileNotFoundError(f'Labels file not found: {labels_path}')
train_df = pd.read_csv(labels_path)
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

all_types = train_df['type'].unique()
type2idx  = {t: i + 1 for i, t in enumerate(sorted(all_types))}
type2idx['<UNK>'] = 0
CFG.N_DOC_TYPES = len(type2idx) + 1
print(f'Recomputed N_DOC_TYPES: {CFG.N_DOC_TYPES} (must match checkpoint embedding size)')

fraud_rate = float(train_df['label'].mean())
print(f'fraud_rate (fill_value, held constant across ALL submissions here): {fraud_rate:.4f}')

# Test images
actual_test_dir, test_files = find_images_robust(data_dir, CFG.TEST_IMG_SUBDIR)
if not test_files:
    raise RuntimeError(f'No test images found under {data_dir / CFG.TEST_IMG_SUBDIR}')
rel_prefix = actual_test_dir.relative_to(data_dir)
print(f'Test images found: {len(test_files)} in {actual_test_dir}')

# Sample submission (template for full 142,818-row output)
sub_csv = data_dir / 'sample_submission.csv'
sub_df  = pd.read_csv(sub_csv) if sub_csv.exists() else pd.DataFrame()
if sub_df.empty:
    raise RuntimeError(f'sample_submission.csv not found at {sub_csv}')
print(f'Sample submission: {len(sub_df)} rows, columns: {sub_df.columns.tolist()}')


def build_test_df(is_digital_value: int) -> pd.DataFrame:
    """Builds test_df identically to v4, except is_digital is fixed to the given value
    for every row instead of hardcoded 0. type_idx stays 0 (<UNK>) either way, matching
    v4 behavior — the private test set includes 2 unseen doc types by design."""
    return pd.DataFrame({
        'id':         [p.stem for p in test_files],
        'image_path': [str(rel_prefix / p.name) for p in test_files],
        'is_digital': is_digital_value,
        'type_idx':   0,
    })


test_df_digital1 = build_test_df(is_digital_value=1)   # diagnostic variant
test_df_digital0 = build_test_df(is_digital_value=0)   # baseline replica (v4's original behavior)
print(f'Built two test_df variants, {len(test_df_digital1)} rows each, '
      f'differing only in is_digital.')

## 5. Transforms (val + TTA only — no training augmentation needed here)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_val_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

TTA_TRANSFORMS = [
    build_val_transform(),
    A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]),
]
print(f'Transforms defined. IMG_SIZE={CFG.IMG_SIZE}, {len(TTA_TRANSFORMS)} TTA variants '
      '(original + horizontal flip — matches v4).')

## 6. Dataset Class

Same `_resolve` path-resolution logic as v4. No face occlusion (train-only in v4, irrelevant for inference).

In [ ]:
class FREUIDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path,
                 transform: Optional[A.Compose] = None,
                 is_train: bool = False) -> None:
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform
        self.is_train  = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, rel_path: str) -> Path:
        c = self.data_dir / rel_path
        if c.exists(): return c
        parts = Path(rel_path).parts
        if len(parts) > 1:
            d = self.data_dir / parts[0] / rel_path
            if d.exists(): return d
        f = self.data_dir / Path(rel_path).name
        if f.exists(): return f
        return c

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        img_path = self._resolve(row['image_path'])
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Warning: cannot load {img_path}: {e}')
            image = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
        if self.transform is not None:
            image = self.transform(image=image)['image']
        out = {
            'image':      image,
            'is_digital': torch.tensor(float(row.get('is_digital', 0)), dtype=torch.float32),
            'type_idx':   torch.tensor(int(row.get('type_idx', 0)), dtype=torch.long),
            'id':         str(row['id']),
        }
        if self.is_train:
            out['label'] = torch.tensor(int(row['label']), dtype=torch.long)
        return out

print('FREUIDDataset defined.')

## 7. Model Architecture (identical to v4 — required for state_dict compatibility)

In [ ]:
class FREUIDModel(nn.Module):
    def __init__(
        self,
        backbone_name: str   = CFG.BACKBONE,
        pretrained:    bool  = CFG.PRETRAINED,
        weights_path:  str   = CFG.WEIGHTS_PATH,
        n_doc_types:   int   = CFG.N_DOC_TYPES,
        doc_emb_dim:   int   = CFG.DOC_EMB_DIM,
        drop_rate:     float = CFG.DROP_RATE,
        use_metadata:  bool  = CFG.USE_METADATA,
    ) -> None:
        super().__init__()
        self.use_metadata = use_metadata

        self.backbone = timm.create_model(
            backbone_name, pretrained=False, num_classes=0, global_pool='avg')

        if pretrained:
            wp = Path(weights_path)
            if wp.exists():
                state_dict = torch.load(wp, map_location='cpu', weights_only=False)
                self.backbone.load_state_dict(state_dict, strict=False)
                print(f'  Backbone ImageNet weights loaded: {wp.name} '
                      f'(will be overwritten by checkpoint state_dict below)')
            else:
                print(f'  NOTE: {weights_path} not found — skipping, checkpoint state_dict '
                      f'will fully populate the model anyway.')

        feat_dim = self.backbone.num_features

        if use_metadata:
            self.doc_embedding = nn.Embedding(
                num_embeddings=n_doc_types, embedding_dim=doc_emb_dim, padding_idx=0)
            meta_dim = doc_emb_dim + 1
        else:
            meta_dim = 0

        in_dim    = feat_dim + meta_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(drop_rate),
            nn.Linear(in_dim, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2), nn.Linear(256, 1),
        )

    def forward(self, image: torch.Tensor,
                is_digital: torch.Tensor,
                type_idx: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(image)
        if self.use_metadata:
            feats = torch.cat(
                [feats, self.doc_embedding(type_idx), is_digital.unsqueeze(1)], dim=1)
        return self.head(feats).squeeze(1)

print('FREUIDModel defined.')

## 8. Calibration Classes

Refit from each checkpoint's stored `val_logits`/`val_labels` so calibration state exactly matches what was used for the real v4 submission.

In [ ]:
class TemperatureScaler:
    def __init__(self): self.temperature = 1.0

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'TemperatureScaler':
        def nll(t):
            t = float(t[0])
            if t <= 0: return 1e9
            p = np.clip(1.0 / (1.0 + np.exp(-logits / t)), 1e-7, 1 - 1e-7)
            return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        res = scipy.optimize.minimize(nll, [1.0], method='L-BFGS-B', bounds=[(0.05, 20.0)])
        self.temperature = float(res.x[0])
        print(f'  Temperature = {self.temperature:.4f}')
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        return (1.0 / (1.0 + np.exp(-logits / self.temperature))).astype(np.float32)


class IsotonicCalibrator:
    def __init__(self): self.iso = IsotonicRegression(out_of_bounds='clip')

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'IsotonicCalibrator':
        scores = 1.0 / (1.0 + np.exp(-logits))
        self.iso.fit(scores, labels)
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        scores = 1.0 / (1.0 + np.exp(-logits))
        return self.iso.predict(scores).astype(np.float32)


def get_calibrator():
    if CFG.CALIB_METHOD == 'temperature': return TemperatureScaler()
    if CFG.CALIB_METHOD == 'isotonic':    return IsotonicCalibrator()
    raise ValueError(f'Unknown calibration method: {CFG.CALIB_METHOD}')

print('TemperatureScaler, IsotonicCalibrator, get_calibrator defined.')

## 9. Inference and Submission

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, df: pd.DataFrame, calibrator=None, use_tta: bool = True) -> np.ndarray:
    model.eval()
    transforms = TTA_TRANSFORMS if use_tta else [build_val_transform()]
    all_logits = []
    for tfm in transforms:
        ds = FREUIDDataset(df, data_dir, tfm, is_train=False)
        loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                             num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
        tfm_logits = []
        for batch in loader:
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                out = model(batch['image'].to(DEVICE),
                            batch['is_digital'].to(DEVICE),
                            batch['type_idx'].to(DEVICE))
            tfm_logits.append(out.cpu().float().numpy())
        if not tfm_logits:
            raise ValueError(f'DataLoader produced 0 batches for {len(df)} rows.')
        all_logits.append(np.concatenate(tfm_logits))

    mean_logits = np.mean(all_logits, axis=0)
    if calibrator is not None:
        return calibrator.transform(mean_logits)
    return (1.0 / (1.0 + np.exp(-mean_logits))).astype(np.float32)


def rank_average(score_arrays: List[np.ndarray]) -> np.ndarray:
    import scipy.stats
    ranks = [scipy.stats.rankdata(s) / len(s) for s in score_arrays]
    return np.mean(ranks, axis=0).astype(np.float32)


def ensemble_predict(fold_results: List, df: pd.DataFrame, method: str = 'rank_avg') -> np.ndarray:
    all_scores = [predict(model, df, calibrator, use_tta=True) for model, calibrator in fold_results]
    if method == 'rank_avg':
        return rank_average(all_scores)
    return np.mean(all_scores, axis=0).astype(np.float32)


def generate_submission(scores: np.ndarray, test_df_: pd.DataFrame,
                         filename: str, fill_value: float) -> pd.DataFrame:
    """Identical logic to v4's generate_submission — fill_value must be passed explicitly
    and kept IDENTICAL across every submission generated in this notebook, per the LB-isolation
    caveat in the header cell."""
    assert len(scores) == len(test_df_), f'Length mismatch: {len(scores)} vs {len(test_df_)}'
    assert np.all((scores >= 0) & (scores <= 1)), 'Scores must be in [0,1]'

    score_col = [c for c in sub_df.columns if c != 'id']
    if not score_col:
        raise ValueError(f'No score column found: {sub_df.columns.tolist()}')
    score_col = score_col[0]

    score_map           = dict(zip(test_df_['id'].astype(str), scores))
    full_sub             = sub_df.copy()
    full_sub[score_col]  = full_sub['id'].astype(str).map(score_map).fillna(fill_value)

    n_real    = full_sub['id'].astype(str).isin(score_map).sum()
    n_default = len(full_sub) - n_real

    out_path = Path(CFG.OUTPUT_DIR) / filename
    full_sub.to_csv(out_path, index=False)
    print(f'Saved: {out_path} | rows={len(full_sub)} | real={n_real} | '
          f'fill({fill_value:.4f})={n_default}')
    print(f'Real-row scores: min={scores.min():.4f} max={scores.max():.4f} mean={scores.mean():.4f}')
    return full_sub

print('predict(), rank_average(), ensemble_predict(), generate_submission() defined.')

## 10. Load Fold Checkpoints + Refit Calibrators

No training happens here — this only loads the frozen `model_state` dict from each checkpoint and refits the (deterministic) calibrator from the checkpoint's own stored `val_logits`/`val_labels`, exactly reproducing the calibration used for the real v4 submission.

In [ ]:
fold_ckpt_paths = sorted(Path(CFG.CHECKPOINT_DIR).glob('fold*_best.pth'))
if not fold_ckpt_paths:
    raise RuntimeError(f'No fold checkpoints found in {CFG.CHECKPOINT_DIR}. '
                        f'Check Section 3 (Checkpoint Restore).')
print(f'Found {len(fold_ckpt_paths)} fold checkpoint(s): {[p.name for p in fold_ckpt_paths]}')

fold_results = []   # list of (model, calibrator)
for ckpt_path in fold_ckpt_paths:
    print(f'\nLoading {ckpt_path.name}...')
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    model = FREUIDModel().to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    calibrator = get_calibrator()
    calibrator.fit(ckpt['val_logits'], ckpt['val_labels'])

    print(f'  Checkpoint FREUID (stored): {ckpt["val_metrics"]["freuid"]:.4f} '
          f'at epoch {ckpt["epoch"]}, fold {ckpt.get("fold", "?")}')
    fold_results.append((model, calibrator))

print(f'\n{len(fold_results)} fold model(s) loaded and calibrated. Ready for inference.')

## 11. EXECUTION — Run Both Variants, Diff Locally, Save Both Files

**This is the only cell that runs heavy computation.** Both variants use the identical `fold_results` (same weights, same calibrators) and the identical `fill_value` — the *only* difference between the two runs is the `is_digital` column on the 7,821 public test rows.

In [ ]:
print('=== Pre-flight ===')
print(f'Test rows          : {len(test_df_digital1)}')
print(f'Fold models loaded  : {len(fold_results)}')
print(f'fill_value (fixed)  : {fraud_rate:.4f}  <-- identical for both submissions below')
print()

# ── Variant B: the diagnostic — is_digital=1 for all test rows ────────────
print('Running ensemble inference: is_digital=1 (diagnostic variant)...')
t0 = time.time()
scores_digital1 = ensemble_predict(fold_results, test_df_digital1, method='rank_avg')
print(f'  done in {time.time()-t0:.0f}s')

sub_digital1 = generate_submission(
    scores_digital1, test_df_digital1,
    filename='submission_diag_isdigital1_ensemble.csv',
    fill_value=fraud_rate,
)

# ── Variant A: baseline replica — is_digital=0, matches original v4 submission ──
print('\nRunning ensemble inference: is_digital=0 (baseline replica, local diff only)...')
t0 = time.time()
scores_digital0 = ensemble_predict(fold_results, test_df_digital0, method='rank_avg')
print(f'  done in {time.time()-t0:.0f}s')

sub_digital0 = generate_submission(
    scores_digital0, test_df_digital0,
    filename='submission_diag_isdigital0_baseline.csv',
    fill_value=fraud_rate,
)

# ── Local diff report — a free signal before spending a Kaggle submission ──
print('\n' + '=' * 60)
print('LOCAL DIFF REPORT (is_digital=1 vs is_digital=0), public rows only')
print('=' * 60)
delta = scores_digital1 - scores_digital0
abs_delta = np.abs(delta)
corr = float(np.corrcoef(scores_digital1, scores_digital0)[0, 1])
crossed_05 = int(np.sum((scores_digital0 < 0.5) != (scores_digital1 < 0.5)))

print(f'Mean score (is_digital=0) : {scores_digital0.mean():.4f}')
print(f'Mean score (is_digital=1) : {scores_digital1.mean():.4f}')
print(f'Mean |delta|              : {abs_delta.mean():.4f}')
print(f'Max  |delta|               : {abs_delta.max():.4f}')
print(f'Pearson correlation        : {corr:.4f}')
print(f'Rows crossing 0.5 threshold: {crossed_05} / {len(scores_digital0)} '
      f'({crossed_05/len(scores_digital0):.1%})')

if abs_delta.mean() < 0.01:
    print('\n>> Scores barely moved. The is_digital flag is likely NOT a major driver of the '
          'CV/LB gap — look elsewhere (unseen doc types, distributional drift, etc.) before '
          'spending a submission on this variant.')
else:
    print('\n>> Scores moved meaningfully. Worth spending a submission on '
          'submission_diag_isdigital1_ensemble.csv to see if public LB improves.')

print('\n' + '=' * 60)
print('SUBMIT TO KAGGLE:  submission_diag_isdigital1_ensemble.csv  (ONLY this one)')
print('DO NOT SUBMIT:     submission_diag_isdigital0_baseline.csv  (duplicates prior submission)')
print('=' * 60)